# Поиск признаков по похожим клиентам

In [ ]:
'''
Пайплайн с признаками v2: CV 0.7397, Public 0.72933. Модели почти одинаковы (корреляция 0.995+),
подбор параметров даёт +0.002 — прирост может дать только новая информация в признаках.
Главный признак — время до момента, когда пауза сравняется с самой длинной паузой за год.
Собственных пауз у клиента мало (у трети клиентов их нет вовсе), поэтому здесь проверяем признаки,
которые берут распределение пауз у всех клиентов:
  P — вероятность следующего заказа в окне по таблице дожития (Каплан — Мейер) по всем клиентам,
      в разрезе числа дней с заказами, прошлой паузы и самой длинной прошлой паузы;
  T — доля target = 1 в ячейке «давность × число дней × запас до самой длинной паузы» (вне фолда);
  M — родственники «самой длинной паузы» на уровне дней.
Каждая группа добавляется к 315 признакам v2, LightGBM с параметрами из пайплайна, 5 фолдов.
Нужен вход: вывод ноутбука с пайплайном (features_v2.parquet и features_v2_names.json).
'''
import json
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("LAMODA_OUT_DIR", "."))
found = sorted(Path("/kaggle/input").rglob("features_v2.parquet")) if Path("/kaggle/input").exists() else []
FEAT_PATH = found[0] if found else OUT_DIR / "features_v2.parquet"
print("признаки v2:", FEAT_PATH)

SEED = 42
GAP_DAYS = 14
N_FOLDS = 5
LGB = dict(objective="binary", learning_rate=0.03, n_estimators=5000, subsample_freq=1, cat_smooth=50, n_jobs=-1, verbose=-1,
           num_leaves=47, min_child_samples=85, colsample_bytree=0.60, subsample=0.89, reg_lambda=0.66, reg_alpha=0.0046,
           min_split_gain=0.43, max_bin=255)
T0 = time.time()


def fast_ap(target, score):
    order = np.argsort(-np.asarray(score), kind="mergesort")
    t = np.asarray(target)[order]
    k = int(np.ceil(len(t) * 0.20))
    top = t[:k]
    return float((np.cumsum(top) / np.arange(1, k + 1) * top).sum() / max(min(k, int(t.sum())), 1))


data = pd.read_parquet(FEAT_PATH)
meta = json.load(open(FEAT_PATH.parent / "features_v2_names.json"))
NAMES, FEATURES = meta["names"], meta["features"]
INV = {v: k for k, v in NAMES.items()}
if "top_category2" in data:
    data["top_category2"] = data["top_category2"].astype("category")
data["scoring_date"] = pd.to_datetime(data["slice"])
data["cutoff"] = data["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
data["win_len"] = data["scoring_date"].map(lambda d: (d + pd.DateOffset(months=3) - d).days)

orders = pd.read_parquet(DATA_DIR / "orders.parquet", columns=["customer_id", "order_id", "order_date"])
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders = orders.merge(data[["customer_id", "cutoff"]], on="customer_id", validate="m:1")
orders["d"] = (orders["cutoff"] - orders["order_date"]).dt.days
print("клиентов:", len(data), "| признаков v2:", len(FEATURES), f"| {(time.time() - T0) / 60:.1f} мин")

## Таблица дожития по всем клиентам

In [ ]:
cid = "customer_id"
info = data.set_index(cid)
R = info["recency"].astype(int)
LO = R + GAP_DAYS
HI = R + GAP_DAYS + info["win_len"]

# дни с заказами: от старого к свежему; у каждого дня — длительность паузы после него
dy = orders[[cid, "d"]].drop_duplicates().sort_values([cid, "d"], ascending=[True, False]).reset_index(drop=True)
dy["j"] = dy.groupby(cid).cumcount() + 1                       # номер дня с заказом у клиента
dy["gap_in"] = -dy.groupby(cid)["d"].diff()                    # пауза, которая закончилась этим днём
dy["pmax"] = dy.groupby(cid)["gap_in"].cummax()                # самая длинная пауза к этому дню
nxt = dy.groupby(cid)["d"].shift(-1)
dy["ev"] = nxt.notna().astype(int)                             # 1 — следующий день известен, 0 — пауза ещё идёт
dy["dur"] = np.where(dy["ev"] == 1, dy["d"] - nxt, dy["d"]).astype(int)
last = dy[dy["ev"] == 0].set_index(cid)                        # текущее состояние клиента
print("дней с заказами:", len(dy), "| известных пауз:", int(dy["ev"].sum()))

G = 1000
kb = np.digitize(dy["j"], [2, 3, 4, 5, 7, 10, 15])                                  # 8 групп
pgb = np.where(dy["gap_in"].isna(), 0, np.digitize(dy["gap_in"].fillna(0), [8, 31, 91, 181]) + 1)   # 6 групп
mgb = np.where(dy["pmax"].isna(), 0, np.digitize(dy["pmax"].fillna(0), [31, 91, 181, 366]) + 1)    # 6 групп
KEYS = {"дни": kb, "дни×пауза": kb * 6 + pgb, "дни×макс": kb * 6 + mgb, "дни×пауза×макс": (kb * 6 + pgb) * 6 + mgb}


def km(key, dur, ev, nk):
    # S[k, t] = P(пауза > t) в группе k с учётом незаконченных пауз
    idx = key * G + np.clip(dur, 0, G - 1)
    ends = np.bincount(idx, minlength=nk * G).reshape(nk, G)
    evs = np.bincount(idx, weights=ev, minlength=nk * G).reshape(nk, G)
    risk = ends[:, ::-1].cumsum(axis=1)[:, ::-1]
    h = np.divide(evs, risk, out=np.zeros_like(evs), where=risk > 0)
    return np.cumprod(1 - h, axis=1)


P = pd.DataFrame(index=info.index)
pos = dy.index[dy["ev"] == 0]
lc = dy.loc[pos, cid].to_numpy()
r_, lo_, hi_ = R.loc[lc].to_numpy(), LO.loc[lc].to_numpy(), np.minimum(HI.loc[lc].to_numpy(), G - 1)
for name, key in KEYS.items():
    S = km(key, dy["dur"].to_numpy(), dy["ev"].to_numpy().astype(float), int(key.max()) + 1)
    k = key[pos]
    s0, s1, s2 = S[k, np.maximum(r_ - 1, 0)], S[k, lo_], S[k, hi_]        # P(пауза ≥ r), P(> lo), P(> hi)
    s0 = np.where(r_ == 0, 1.0, s0)
    col = pd.Series(lc)
    P[f"дожитие|{name}|в окне"] = pd.Series((s1 - s2) / np.maximum(s0, 1e-9), index=lc)
    P[f"дожитие|{name}|до окна"] = pd.Series((s0 - s1) / np.maximum(s0, 1e-9), index=lc)
    P[f"дожитие|{name}|позже"] = pd.Series(s2 / np.maximum(s0, 1e-9), index=lc)
    P[f"дожитие|{name}|выжило до r"] = pd.Series(s0, index=lc)

# то же, но паузы в единицах собственной медианной паузы клиента
med = dy.groupby(cid)["gap_in"].median()
rel = dy[cid].map(med)
ok = rel.notna().to_numpy() & (rel.to_numpy() > 0)
scale = 20.0                                                   # 1 единица медианы = 20 клеток сетки
dur_rel = np.round(dy["dur"].to_numpy() / np.where(ok, rel.to_numpy(), 1) * scale).astype(int)
key = kb[ok]
S = km(key, dur_rel[ok], dy["ev"].to_numpy()[ok].astype(float), 8)
m_c = med.reindex(lc).to_numpy()
okc = ~np.isnan(m_c) & (m_c > 0)
kk = kb[pos]
def at(x):
    return S[kk, np.clip(np.round(x / np.where(okc, m_c, 1) * scale).astype(int), 0, G - 1)]
s0, s1, s2 = at(np.maximum(r_ - 1, 0)), at(lo_), at(hi_)
P["дожитие|в медианах|в окне"] = pd.Series(np.where(okc, (s1 - s2) / np.maximum(s0, 1e-9), np.nan), index=lc)
P["дожитие|в медианах|позже"] = pd.Series(np.where(okc, s2 / np.maximum(s0, 1e-9), np.nan), index=lc)
print(f"группа P: {P.shape[1]} признаков | {(time.time() - T0) / 60:.1f} мин")
display(P.join(info["target"]).groupby(pd.qcut(P["дожитие|дни×пауза×макс|в окне"], 10, duplicates="drop"), observed=True)["target"].agg(["mean", "size"]).T)

## Самая длинная пауза и её родственники

In [ ]:
gp = dy[dy["gap_in"].notna()]
M = pd.DataFrame(index=info.index)
r = R.astype(float)
win = info["win_len"].astype(float)
for hname, H in [("вся", 10_000), ("год", 365)]:
    g = gp[gp["d"] <= H].groupby(cid)["gap_in"]
    top = gp[gp["d"] <= H].sort_values([cid, "gap_in"], ascending=[True, False]).groupby(cid)["gap_in"]
    mg, mg2 = g.max().reindex(info.index), top.nth(1).groupby(gp.loc[top.nth(1).index, cid]).first().reindex(info.index)
    mg3 = top.nth(2).groupby(gp.loc[top.nth(2).index, cid]).first().reindex(info.index)
    for nm, v in [("макс", mg), ("2-я", mg2), ("3-я", mg3)]:
        M[f"пауза|{hname}|{nm}"] = v
        M[f"пауза|{hname}|{nm} - r"] = v - r
        M[f"пауза|{hname}|r / {nm}"] = r / v.replace(0, np.nan)
        M[f"пауза|{hname}|{nm}: место в окне"] = ((v - r - GAP_DAYS) / win).clip(-1, 2)
    M[f"пауза|{hname}|пауз ≥ r"] = (gp["gap_in"] >= gp[cid].map(r)).where(gp["d"] <= H).groupby(gp[cid]).sum().reindex(info.index).fillna(0)
    M[f"пауза|{hname}|пауз в окне"] = ((gp["gap_in"] > gp[cid].map(LO)) & (gp["gap_in"] <= gp[cid].map(HI))).where(gp["d"] <= H).groupby(gp[cid]).sum().reindex(info.index).fillna(0)
M["пауза|r - макс до последнего года"] = r - gp[gp["d"] > 365].groupby(cid)["gap_in"].max().reindex(info.index)
M["пауза|последняя / макс"] = last["gap_in"].reindex(info.index) / M["пауза|вся|макс"].replace(0, np.nan)
M["пауза|макс растёт"] = (last["gap_in"].reindex(info.index) >= last["pmax"].reindex(info.index)).astype(float).where(last["gap_in"].reindex(info.index).notna())
print(f"группа M: {M.shape[1]} признаков | {(time.time() - T0) / 60:.1f} мин")

## Доля покупателей в похожих ячейках

In [ ]:
tr = data[data["part"].eq("train")].reset_index(drop=True)
y = tr["target"].astype(int).to_numpy()
sl = tr["slice"].to_numpy()
strata = tr["slice"] + "_" + tr["target"].astype(int).astype(str)
SPLIT = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(tr, strata))

mx = info[INV["заказ|год|max|t"]] if INV.get("заказ|год|max|t") in info else M["пауза|год|макс - r"]
cells = {
    "давность×дни": [pd.qcut(info["recency"], 20, labels=False, duplicates="drop"), np.digitize(info["n_days"], [2, 3, 4, 6, 10, 21])],
    "давность×дни×запас": [pd.qcut(info["recency"], 12, labels=False, duplicates="drop"), np.digitize(info["n_days"], [2, 3, 4, 6, 10, 21]),
                           pd.qcut(mx.fillna(-9999), 8, labels=False, duplicates="drop")],
    "запас×дни×заказов за год": [pd.qcut(mx.fillna(-9999), 15, labels=False, duplicates="drop"), np.digitize(info["n_days"], [2, 3, 4, 6, 10, 21]),
                                 np.digitize(info["n_orders_365"], [2, 3, 5, 8, 13])],
}
T = pd.DataFrame(index=info.index)
for name, parts in cells.items():
    key = pd.Series(np.zeros(len(info), dtype=np.int64), index=info.index)
    for p in parts:
        key = key * 100 + pd.Series(np.asarray(p), index=info.index).fillna(99).astype(np.int64)
    k_tr = key.loc[tr[cid]].to_numpy()
    prior, m = y.mean(), 30.0
    oof = np.zeros(len(tr))
    for a, b in SPLIT:
        st = pd.DataFrame({"k": k_tr[a], "y": y[a]}).groupby("k")["y"].agg(["sum", "count"])
        s = pd.Series(k_tr[b]).map((st["sum"] + m * prior) / (st["count"] + m)).fillna(prior)
        oof[b] = s.to_numpy()
    st = pd.DataFrame({"k": k_tr, "y": y}).groupby("k")["y"].agg(["sum", "count"])
    full = key.map((st["sum"] + m * prior) / (st["count"] + m)).fillna(prior)
    full.loc[tr[cid]] = oof
    T[f"ячейка|{name}"] = full
print(f"группа T: {T.shape[1]} признаков | {(time.time() - T0) / 60:.1f} мин")

## Сравнение групп

In [ ]:
GROUPS = {"P": P, "M": M, "T": T}
ALL = pd.concat(GROUPS.values(), axis=1).astype("float32")
SHORT = {f"q{i}": c for i, c in enumerate(ALL.columns)}
ALL.columns = list(SHORT)
GCOLS = {g: [f"q{i}" for i, c in enumerate(pd.concat(GROUPS.values(), axis=1).columns) if c in set(df.columns)] for g, df in GROUPS.items()}
Xall = tr[[cid] + FEATURES].join(ALL, on=cid).drop(columns=cid)


def score(pred):
    return float(np.mean([fast_ap(y[sl == s], pred[sl == s]) for s in np.unique(sl)]))


def cv(cols):
    oof, imp = np.zeros(len(tr)), pd.Series(0.0, index=cols)
    for a, b in SPLIT:
        m = lgb.LGBMClassifier(**LGB, random_state=SEED).fit(Xall[cols].iloc[a], y[a], eval_set=[(Xall[cols].iloc[b], y[b])],
                                                            callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[b] = m.predict_proba(Xall[cols].iloc[b])[:, 1]
        imp += m.booster_.feature_importance("gain")
    return oof, imp / imp.sum()


rows, OOF = [], {}
for name, cols in [("v2", FEATURES)] + [(f"v2 + {g}", FEATURES + c) for g, c in GCOLS.items()] + [("v2 + все", FEATURES + list(SHORT))]:
    t0 = time.time()
    oof, imp = cv(cols)
    OOF[name] = (oof, imp)
    rows.append({"набор": name, "признаков": len(cols), "AP@20%": score(oof),
                 **{s: fast_ap(y[sl == s], oof[sl == s]) for s in np.unique(sl)}, "минут": round((time.time() - t0) / 60, 1)})
    print(rows[-1])
res = pd.DataFrame(rows).set_index("набор")
res["прирост"] = res["AP@20%"] - res.loc["v2", "AP@20%"]
display(res)
imp = OOF["v2 + все"][1]
print("Доля важности новых групп:", {g: round(float(imp[c].sum()), 3) for g, c in GCOLS.items()})
print("Самые важные признаки в наборе «v2 + все»:")
display(imp.sort_values(ascending=False).head(30).rename(index={**NAMES, **SHORT}).to_frame("доля важности"))

new = ALL.copy()
new.columns = [SHORT[c] for c in new.columns]
new.reset_index().to_parquet(OUT_DIR / "features_v3_new.parquet", index=False)
print(f"Сохранено: features_v3_new.parquet. Всего времени: {(time.time() - T0) / 60:.1f} мин")